# Machine Learning 

### Preparación del dataset para clustering

En 02_Analisis_desciptivo.ipynb concluimos que el dataset está dominado por variables binarias y categóricas (34 factores de riesgo 0/1, 7 categóricas, solo 1-2 numéricas reales)
También identificamos que:
- Las 8 columnas de método (ahorcamiento, arma_cortopunzante, etc.) son mejor resumidas en una sola variable categórica metodo_principal.
- Las columnas id_ano, las fechas crudas y barrio_de_residencia (162 categorías tras normalizar) no son atributos de perfil de riesgo, así que tampoco entran al modelo, pero los conservamos aparte para análisis posteriores.
Este notebook parte de data_clean.csv que contiene el dataset limpio y arma dos datasets: uno con las variables para el clustering y otro con las variables de validación/metadata.

#### Cargue de librerías

In [1]:
import pandas as pd
import numpy as np

### Cargar dataset limpio

In [2]:
# Cargamos el dataset ya limpio (barrios normalizados y fechas corregidas)
df = pd.read_csv('data/data_clean.csv', sep=',', encoding='utf-8')

# Guardamos el índice original como id_registro para poder cruzar más adelante
# el resultado del clustering con las variables de validación
df.insert(0, 'id_registro', df.index)

df.head()

,id_registro,id_ano,fecha_de_notificacion_del_evento,sexo,area_de_residencia,barrio_de_residencia,seguridad_social,estrato_socioeconomico,gestante,poblacion_a_cargo_icbf,...,inmolacion,lanzamiento_al_vacio,lanzamiento_a_vehiculo,lanzamiento_al_agua,intoxicaciones,remitido_a_psiquiatria,remitido_a_psicologia,remitido_a_trabajo_social,edad_rango,escolaridad_grupo
0,0,2020,2020-01-02,FEMENINO,CABECERA MUNICIPAL,CASTILLA,CONTRIBUTIVO,3,0,0,...,0,0,0,0,0,1,1,0,adulto_joven,Básica
1,1,2020,2020-01-03,FEMENINO,CABECERA MUNICIPAL,LA MARIA,CONTRIBUTIVO,5,0,0,...,0,0,0,0,1,1,1,0,adulto_mayor,Superior
2,2,2020,2020-01-17,MASCULINO,CABECERA MUNICIPAL,LIBERTADOR,NO ASEGURADO,2,0,0,...,0,1,0,0,0,1,1,0,adulto_mayor,Superior
3,3,2020,2020-01-21,MASCULINO,CABECERA MUNICIPAL,ASIS BOYACENSE,CONTRIBUTIVO,3,0,0,...,0,0,0,0,1,1,1,0,joven,Básica
4,4,2020,2020-01-21,MASCULINO,CABECERA MUNICIPAL,REMANSOS DE LA SABANA,CONTRIBUTIVO,3,0,0,...,0,0,0,0,0,1,0,0,adulto_joven,Superior


#### Reconstruimos metodo_principal

La variable derivada `metodo_principal` que armamos en el análisis descriptivo no se exportó en `data_clean.csv`, así que la reconstruimos aquí.

In [3]:
columnas_metodo_original = ['ahorcamiento', 'arma_cortopunzante', 'arma_de_fuego', 'inmolacion',
                             'lanzamiento_al_vacio', 'lanzamiento_a_vehiculo', 'lanzamiento_al_agua',
                             'intoxicaciones']

def obtener_metodo(row):
    metodos = [col for col in columnas_metodo_original if row[col] == 1]
    if len(metodos) == 0:
        return 'no especificado'
    if len(metodos) > 1:
        return 'multiple'
    return metodos[0]

df['metodo_principal'] = df[columnas_metodo_original].apply(obtener_metodo, axis=1)

print("Distribución de metodo_principal:")
display(df['metodo_principal'].value_counts())

Distribución de metodo_principal:


metodo_principal
intoxicaciones            452
arma_cortopunzante        121
ahorcamiento               79
multiple                   76
lanzamiento_al_vacio       61
lanzamiento_a_vehiculo     24
no especificado             3
arma_de_fuego               1
inmolacion                  1
Name: count, dtype: int64

#### Convertimos las fechas y calculamos el retraso de notificación

Igual que `metodo_principal`, `dias_retraso_notificacion` no quedó en `data_clean.csv`. La calculamos de nuevo, pero la dejamos en el grupo de variables de metadata/validación, no como variable de entrada al clustering (ya que no es un factor de riesgo del perfil).

In [4]:
df['fecha_del_hecho'] = pd.to_datetime(df['fecha_del_hecho'])
df['fecha_de_notificacion_del_evento'] = pd.to_datetime(df['fecha_de_notificacion_del_evento'])
df['dias_retraso_notificacion'] = (df['fecha_de_notificacion_del_evento'] - df['fecha_del_hecho']).dt.days

print("Medidas de tendencia central - días de retraso en la notificación:")
display(df['dias_retraso_notificacion'].agg(["mean", "median", "std", "min", "max"]))

Medidas de tendencia central - días de retraso en la notificación:


mean        4.257946
median      0.000000
std        15.219189
min         0.000000
max       228.000000
Name: dias_retraso_notificacion, dtype: float64

#### Definimos las variables para el modelo de clustering

Separamos las columnas en tres grupos:
- `columnas_metadata`: identificador, año, fechas, retraso y barrio — se conservan para análisis posteriores, no entran al modelo.
- `columnas_validacion`: `numero_de_intentos` — se guarda aparte para validar los clusters, no para formarlos.
- `columnas_modelo`: todo lo demás (demográficas, estrato, los 34 factores de riesgo binarios incluyendo `intentos_previos`, y `metodo_principal`), que sí describen el perfil de riesgo de la persona.

In [5]:
columnas_metadata = ['id_registro', 'id_ano', 'fecha_del_hecho', 'fecha_de_notificacion_del_evento',
                      'dias_retraso_notificacion', 'barrio_de_residencia']
columnas_validacion = ['numero_de_intentos']

columnas_modelo = [col for col in df.columns
                    if col not in columnas_metadata + columnas_validacion + columnas_metodo_original]

print(f"Variables de metadata (no entran al modelo):\n {columnas_metadata} \n")
print(f"Variable de validación (no entra al modelo):\n {columnas_validacion} \n")
print(f"Variables para el modelo de clustering ({len(columnas_modelo)}):\n {columnas_modelo} \n")

Variables de metadata (no entran al modelo):
 ['id_registro', 'id_ano', 'fecha_del_hecho', 'fecha_de_notificacion_del_evento', 'dias_retraso_notificacion', 'barrio_de_residencia'] 

Variable de validación (no entra al modelo):
 ['numero_de_intentos'] 

Variables para el modelo de clustering (35):
 ['sexo', 'area_de_residencia', 'seguridad_social', 'estrato_socioeconomico', 'gestante', 'poblacion_a_cargo_icbf', 'intentos_previos', 'estado_civil', 'conflicto_con_pareja_o_ex_pareja', 'enfermedad_cronica_dolorosa_o_discapacitante', 'problemas_economicos', 'muerte_de_un_familiar', 'escolar_educativa', 'problemas_juridicos', 'suicidio_de_un_familiar', 'maltrato_fisico_sicologico_sexual', 'problemas_laborales', 'problemas_familiares', 'consumo_de_spa', 'antecedentes_familiares_de_conducta_suicida', 'ideacion_suicida_persistente', 'plan_organizado_de_suicidio', 'antecedente_trastorno_psicquiatrico', 'trastorno_depresivo', 'trastorno_de_personalidad', 'trastorno_bipolar', 'esquizofrenia', 'ante

#### Dataset final para clustering

In [6]:
df_cluster = df[['id_registro'] + [col for col in columnas_modelo if col != 'id_registro']]
df_validacion = df[columnas_metadata + columnas_validacion]

print("Dataset para clustering:")
df_cluster.info()

print("\nDataset de validación/metadata:")
df_validacion.info()

Dataset para clustering:
<class 'pandas.DataFrame'>
RangeIndex: 818 entries, 0 to 817
Data columns (total 36 columns):
 #   Column                                        Non-Null Count  Dtype
---  ------                                        --------------  -----
 0   id_registro                                   818 non-null    int64
 1   sexo                                          818 non-null    str  
 2   area_de_residencia                            818 non-null    str  
 3   seguridad_social                              818 non-null    str  
 4   estrato_socioeconomico                        818 non-null    int64
 5   gestante                                      818 non-null    int64
 6   poblacion_a_cargo_icbf                        818 non-null    int64
 7   intentos_previos                              818 non-null    int64
 8   estado_civil                                  818 non-null    str  
 9   conflicto_con_pareja_o_ex_pareja              818 non-null    int64
 10  

#### Exportamos los datasets

In [7]:
df_cluster.to_csv('data/data_cluster.csv', index=False, encoding='utf-8')
df_validacion.to_csv('data/data_validacion.csv', index=False, encoding='utf-8')

In [8]:
df_cluster.head()

,id_registro,sexo,area_de_residencia,seguridad_social,estrato_socioeconomico,gestante,poblacion_a_cargo_icbf,intentos_previos,estado_civil,conflicto_con_pareja_o_ex_pareja,...,trastorno_bipolar,esquizofrenia,antecedente_violencia_o_abuso,abuso_de_alcohol,remitido_a_psiquiatria,remitido_a_psicologia,remitido_a_trabajo_social,edad_rango,escolaridad_grupo,metodo_principal
0,0,FEMENINO,CABECERA MUNICIPAL,CONTRIBUTIVO,3,0,0,1,SOLTERO(A),0,...,0,0,0,0,1,1,0,adulto_joven,Básica,arma_cortopunzante
1,1,FEMENINO,CABECERA MUNICIPAL,CONTRIBUTIVO,5,0,0,0,UNION LIBRE,1,...,0,0,0,0,1,1,0,adulto_mayor,Superior,intoxicaciones
2,2,MASCULINO,CABECERA MUNICIPAL,NO ASEGURADO,2,0,0,1,CASADO(A),1,...,0,0,0,0,1,1,0,adulto_mayor,Superior,lanzamiento_al_vacio
3,3,MASCULINO,CABECERA MUNICIPAL,CONTRIBUTIVO,3,0,0,0,SOLTERO(A),1,...,0,0,0,0,1,1,0,joven,Básica,multiple
4,4,MASCULINO,CABECERA MUNICIPAL,CONTRIBUTIVO,3,0,0,1,SOLTERO(A),0,...,0,0,0,0,1,0,0,adulto_joven,Superior,arma_cortopunzante
